# SEM Microstructure Heat Treatment Prediction - Demo Notebook

This notebook demonstrates the full capabilities of the SEM Microstructure Heat Treatment Prediction pipeline:

1. **Data Loading & Exploration** - Load and explore the microstructure dataset
2. **Tabular Feature Preprocessing** - Handle missing data, encode categorical features, scale numeric features
3. **Image Feature Extraction** - Extract deep features using pre-trained CNN backbones
4. **Model Training & Evaluation** - Train ensemble models and compare performance
5. **Visualization** - Learning curves, model comparison, and predictions

In [ ]:
# Standard imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path for local imports
sys.path.insert(0, os.path.abspath('..'))

# Project imports
from src.preprocessing import FeaturePreprocessor
from src.preprocessing.pipeline import PreprocessingConfig, MissingDataConfig, ScalingConfig, EncodingConfig
from src.extraction import FeatureExtractor, BackboneRegistry
from src.extraction.extractor import ExtractionConfig
from src.model_trainer import (
    ModelTrainer, 
    build_ensemble_models, 
    evaluate_model,
    plot_predictions,
    plot_learning_curves,
    plot_model_comparison
)
from src.config import Config

# Display settings
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Data Loading & Exploration

The microstructure dataset contains:
- **Metadata**: Alloy names, article URLs, sample IDs
- **Chemical Composition**: C, Mn, Si, Cr, P, S, Mo, Cu, Ni, Al, Nb, V, Ti, Fe, B, etc.
- **Heat Treatment Parameters**: Multi-cycle processing (heating rate, holding temp/time, cooling rate, quench type)
- **Image Links**: References to SEM microstructure images

In [ ]:
# Load the microstructure dataset
df = pd.read_csv('../datasets/microstructure.csv', header=1)  # Skip the category header row

print(f"Dataset shape: {df.shape}")
print(f"\nColumn count: {len(df.columns)}")
df.head()

In [ ]:
# Display all column names grouped by category
print("=" * 60)
print("DATASET COLUMNS")
print("=" * 60)

for i, col in enumerate(df.columns):
    print(f"{i:3d}. {col}")

In [ ]:
# Examine data types and missing values
print("Data Types and Missing Values:")
print("-" * 60)

info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null_count': df.isna().sum(),
    'null_pct': (df.isna().sum() / len(df) * 100).round(1)
})

# Show columns with significant data
info_df[info_df['non_null'] > 0].head(30)

In [ ]:
# Visualize missing data patterns
fig, ax = plt.subplots(figsize=(14, 6))

# Calculate missing percentages for first 40 columns
missing_pct = (df.iloc[:, :40].isna().sum() / len(df) * 100)

colors = ['#2ecc71' if x < 50 else '#e74c3c' for x in missing_pct]
bars = ax.bar(range(len(missing_pct)), missing_pct, color=colors)

ax.set_xlabel('Column Index')
ax.set_ylabel('Missing %')
ax.set_title('Missing Data by Column (First 40 Columns)')
ax.axhline(y=50, color='orange', linestyle='--', label='50% threshold')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Examine unique alloy types
print("Alloy Types in Dataset:")
print("-" * 40)
alloy_counts = df['Alloy'].value_counts()
print(alloy_counts)

# Visualize alloy distribution
fig, ax = plt.subplots(figsize=(10, 5))
alloy_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_ylabel('Alloy Type')
ax.set_title('Distribution of Alloy Types')
plt.tight_layout()
plt.show()

## 2. Tabular Feature Preprocessing

The preprocessing pipeline handles:
- **Missing Data**: Column dropping (>95% missing), row filling strategies
- **Type Inference**: Automatic detection of numeric, categorical, text, boolean columns
- **Encoding**: One-hot for categorical, TF-IDF for text
- **Scaling**: Standard, MinMax, or Robust scaling

In [ ]:
# Select feature columns for preprocessing demo
# Focus on chemical composition and key heat treatment parameters

feature_columns = [
    'C (wt.%)', 'Mn (wt.%)', 'Si', 'Cr (wt.%)', 'P', 'S', 'Mo', 'Cu', 'Ni', 'Al',
    'Heat treatment type', 'Num_Cycles',
    'Cycle1_HoldingTemp (C)', 'Cycle1_HoldingTime (min)'
]

# Filter to columns that exist
feature_columns = [c for c in feature_columns if c in df.columns]
print(f"Selected {len(feature_columns)} feature columns:")
for col in feature_columns:
    print(f"  - {col}")

In [ ]:
# Create preprocessing configuration
preproc_config = PreprocessingConfig(
    missing_data=MissingDataConfig(
        column_drop_threshold=0.95,  # Drop columns with >95% missing
        row_fill_threshold=0.10,     # Fill if <10% missing
        numeric_fill_strategy='mean',
        categorical_fill_strategy='mode'
    ),
    scaling=ScalingConfig(
        method='standard',
        enabled=True
    ),
    encoding=EncodingConfig(
        categorical='onehot',
        max_categories=50
    )
)

print("Preprocessing Configuration:")
print(f"  Missing data threshold: {preproc_config.missing_data.column_drop_threshold}")
print(f"  Scaling method: {preproc_config.scaling.method}")
print(f"  Categorical encoding: {preproc_config.encoding.categorical}")

In [ ]:
# Initialize and fit the preprocessor
preprocessor = FeaturePreprocessor(preproc_config)

# Select subset of data for demo
df_features = df[feature_columns].copy()

print("Fitting preprocessor...")
print("=" * 60)
X_tabular = preprocessor.fit_transform(df_features)

In [ ]:
# Examine preprocessed features
print(f"\nPreprocessed feature shape: {X_tabular.shape}")
print(f"\nFeature names ({len(preprocessor.get_feature_names())}):")
for name in preprocessor.get_feature_names()[:20]:
    print(f"  - {name}")
if len(preprocessor.get_feature_names()) > 20:
    print(f"  ... and {len(preprocessor.get_feature_names()) - 20} more")

if preprocessor.get_dropped_columns():
    print(f"\nDropped columns: {preprocessor.get_dropped_columns()}")

In [ ]:
# Visualize feature distributions (first 10 numeric features)
feature_names = preprocessor.get_feature_names()
numeric_features = [f for f in feature_names if not f.startswith('Heat treatment')][:10]

if len(numeric_features) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for idx, feat_name in enumerate(numeric_features):
        feat_idx = feature_names.index(feat_name)
        axes[idx].hist(X_tabular[:, feat_idx], bins=20, color='steelblue', alpha=0.7)
        axes[idx].set_title(feat_name[:15], fontsize=9)
        axes[idx].tick_params(labelsize=8)
    
    # Hide empty subplots
    for idx in range(len(numeric_features), 10):
        axes[idx].set_visible(False)
    
    plt.suptitle('Preprocessed Feature Distributions (Scaled)', fontsize=12)
    plt.tight_layout()
    plt.show()

## 3. Image Feature Extraction

The extraction module supports multiple CNN backbones:
- **Classic CNNs**: VGG16, VGG19, ResNet18/50/101
- **Modern Architectures**: DenseNet121, EfficientNet-B0/B4, ConvNeXt-Tiny
- **Self-supervised**: DINOv2 (ViT-S/B/L)

In [ ]:
# List available backbones
print("Available CNN Backbones:")
print("=" * 40)

available = BackboneRegistry.list_available()
for name in sorted(available):
    print(f"  - {name}")

print(f"\nTotal: {len(available)} backbones available")

In [ ]:
# Demonstrate extraction configuration (without actual images)
extraction_config = ExtractionConfig(
    backbones=['resnet50', 'vgg16'],  # Use multiple backbones
    img_size=224,
    batch_size=16,
    num_workers=2,
    pooling='avg'
)

print("Feature Extraction Configuration:")
print(f"  Backbones: {extraction_config.backbones}")
print(f"  Image size: {extraction_config.img_size}x{extraction_config.img_size}")
print(f"  Batch size: {extraction_config.batch_size}")
print(f"  Pooling: {extraction_config.pooling}")

# Show expected feature dimensions
backbone_dims = {
    'vgg16': 512, 'vgg19': 512,
    'resnet18': 512, 'resnet50': 2048, 'resnet101': 2048,
    'densenet121': 1024,
    'efficientnet_b0': 1280, 'efficientnet_b4': 1792,
    'convnext_tiny': 768,
    'mobilenet_v3': 960,
    'dinov2_vits14': 384, 'dinov2_vitb14': 768, 'dinov2_vitl14': 1024
}

total_dim = sum(backbone_dims.get(b, 0) for b in extraction_config.backbones)
print(f"\nExpected feature dimension: {total_dim}")

In [ ]:
# Demonstrate with synthetic image features (since we don't have actual images)
# In real usage, you would use: extractor.extract_features(image_paths)

n_samples = len(df)
feature_dim = 2560  # resnet50 (2048) + vgg16 (512)

# Simulate extracted features
np.random.seed(42)
X_images = np.random.randn(n_samples, feature_dim).astype(np.float32)

print(f"Simulated image features shape: {X_images.shape}")
print(f"  Samples: {n_samples}")
print(f"  Feature dimension: {feature_dim}")

## 4. Model Training & Evaluation

The training module provides:
- **Ensemble Models**: Random Forest, Gradient Boosting, AdaBoost
- **Multi-output Support**: Native RF multi-output, MultiOutputRegressor wrapper for others
- **Learning Curves**: Track metrics across training iterations
- **Model Comparison**: Evaluate all models on test set

In [ ]:
# Create synthetic target variable for demo
# In real usage, this would be actual heat treatment outcomes or material properties

np.random.seed(42)

# Simulate a target variable (e.g., hardness prediction)
# Based on some chemical composition features
target_columns = ['Predicted_Hardness']

# Create synthetic target correlated with features
Y = 200 + 50 * X_tabular[:, 0] + 30 * np.random.randn(n_samples)
Y = Y.reshape(-1, 1).astype(np.float32)

print(f"Target shape: {Y.shape}")
print(f"Target columns: {target_columns}")
print(f"Target range: [{Y.min():.1f}, {Y.max():.1f}]")

In [ ]:
# Combine image and tabular features
X_combined = np.concatenate([X_images, X_tabular], axis=1)

print(f"Combined feature matrix shape: {X_combined.shape}")
print(f"  Image features: {X_images.shape[1]}")
print(f"  Tabular features: {X_tabular.shape[1]}")
print(f"  Total: {X_combined.shape[1]}")

In [ ]:
# Create a minimal config for the trainer
config = Config(
    random_seed=42,
    model_dir='../models'
)

# Initialize trainer with configurable n_estimators
trainer = ModelTrainer(config, n_estimators=100)  # Reduced for faster demo

print(f"Model Trainer initialized:")
print(f"  Random seed: {config.random_seed}")
print(f"  N estimators: {trainer.n_estimators}")
print(f"  Model directory: {config.model_dir}")

In [ ]:
# Split data into train/val/test sets
X_train, X_val, X_test, Y_train, Y_val, Y_test = trainer.split_data(
    X_combined, Y,
    test_size=0.15,
    val_size=0.15
)

print(f"\nData splits:")
print(f"  Train: {X_train.shape}")
print(f"  Val: {X_val.shape}")
print(f"  Test: {X_test.shape}")

In [ ]:
# Train all models and track learning curves
print("\nTraining ensemble models...")
print("=" * 60)

best_model_name = trainer.train_and_evaluate(
    X_train, Y_train,
    X_val, Y_val,
    target_columns,
    track_learning_curves=True  # Enable learning curve tracking
)

In [ ]:
# Plot learning curves for boosting models
print("\nLearning Curves:")
print("=" * 60)

trainer.plot_learning_curves(show=True)

In [ ]:
# Evaluate all models on test set
test_results = trainer.evaluate_all_on_test(X_test, Y_test, target_columns)

In [ ]:
# Plot model comparison
trainer.plot_test_comparison(show=True)

## 5. Predictions Visualization

Visualize predicted vs actual values and analyze model performance.

In [ ]:
# Get predictions from best model
Y_pred = trainer.best_model.predict(X_test)

if Y_pred.ndim == 1:
    Y_pred = Y_pred.reshape(-1, 1)

print(f"Predictions shape: {Y_pred.shape}")

In [ ]:
# Plot predicted vs actual
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(Y_test, Y_pred, alpha=0.6, edgecolors='white', linewidth=0.5)

# Identity line
mn = min(Y_test.min(), Y_pred.min())
mx = max(Y_test.max(), Y_pred.max())
ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Actual Values', fontsize=12)
ax.set_ylabel('Predicted Values', fontsize=12)
ax.set_title(f'Predicted vs Actual ({trainer.best_model_name})', fontsize=14)
ax.legend()

# Add R2 annotation
from sklearn.metrics import r2_score
r2 = r2_score(Y_test, Y_pred)
ax.annotate(f'R² = {r2:.4f}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis
residuals = Y_test.flatten() - Y_pred.flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residual distribution
axes[0].hist(residuals, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution')

# Residuals vs predicted
axes[1].scatter(Y_pred, residuals, alpha=0.6)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Predicted')

plt.tight_layout()
plt.show()

print(f"Residual Statistics:")
print(f"  Mean: {residuals.mean():.4f}")
print(f"  Std: {residuals.std():.4f}")
print(f"  Min: {residuals.min():.4f}")
print(f"  Max: {residuals.max():.4f}")

## 6. Summary

This notebook demonstrated the full pipeline capabilities:

1. **Data Loading**: Loaded microstructure dataset with chemical composition and heat treatment parameters
2. **Preprocessing**: Handled missing data, encoded categoricals, scaled numeric features
3. **Feature Extraction**: Showed available CNN backbones for image feature extraction
4. **Model Training**: Trained RF, GBR, and AdaBoost with learning curve tracking
5. **Evaluation**: Compared models and visualized predictions

### Key Capabilities:
- Extensible backbone registry (12+ CNN architectures)
- Automatic column type inference
- Configurable preprocessing pipeline
- Multi-output regression support
- Learning curve visualization
- Model comparison plots

In [ ]:
# Final summary
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"\nDataset: {n_samples} samples")
print(f"Tabular features: {X_tabular.shape[1]}")
print(f"Image features: {X_images.shape[1]}")
print(f"Total features: {X_combined.shape[1]}")
print(f"\nBest model: {trainer.best_model_name}")
print(f"Test R2: {test_results[trainer.best_model_name]['R2']:.4f}")
print(f"Test MAE: {test_results[trainer.best_model_name]['MAE']:.4f}")
print(f"Test RMSE: {test_results[trainer.best_model_name]['RMSE']:.4f}")
print("=" * 60)